# 02 — Prepare Labels
Merge ECOSoundSet and InsectSet459, balance classes, and produce the `train.csv` / `val.csv` / `test.csv` splits that OpenSoundscape expects.

**Kernel:** `Python (orthoptera-training)`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent  # cwd relative to current notebook path
ECO_ROOT    = PROJECT_ROOT / "datasets" / "ecosoundset"
INSECT_ROOT = PROJECT_ROOT / "datasets" / "insectset459"
OUT_DIR     = PROJECT_ROOT / "training" / "data"
OUT_DIR.mkdir(exist_ok=True)

# Canonical species names as they appear in ECOSoundSet (trinomial).
# InsectSet459 uses underscore format — mapped below.
# Meconema thalassinum is absent from both datasets.
UK_SPECIES = [
    "Chorthippus brunneus brunneus",          # Field Grasshopper
    "Pseudochorthippus parallelus parallelus", # Meadow Grasshopper
    "Omocestus viridulus",                    # Common Green Grasshopper
    "Tettigonia viridissima",                 # Great Green Bush-cricket
    "Roeseliana roeselii",                    # Roesel's Bush-cricket
    "Pholidoptera griseoaptera",              # Dark Bush-cricket
    "Leptophyes punctatissima",               # Speckled Bush-cricket
    "Gryllus campestris",                     # Field Cricket
]

# InsectSet459 species_name uses underscores and may be binomial.
# Map species_name from InsectSet459 to ECO canonical name.
INSECT_TO_ECO = {
    "Chorthippus_brunneus":                   "Chorthippus brunneus brunneus",
    "Pseudochorthippus_parallelus":           "Pseudochorthippus parallelus parallelus",
    "Omocestus_viridulus":                    "Omocestus viridulus",
    "Tettigonia_viridissima":                 "Tettigonia viridissima",
    "Roeseliana_roeselii":                    "Roeseliana roeselii",
    "Pholidoptera_griseoaptera":              "Pholidoptera griseoaptera",
    "Leptophyes_punctatissima":               "Leptophyes punctatissima",
    "Gryllus_campestris":                     "Gryllus campestris"
}


In [22]:
# ── Load ECOSoundSet and split by subset column ───────────────────────────────
all_annot_eco = pd.read_csv(ECO_ROOT / "annotated_audio_segments.csv")
orth_eco = all_annot_eco[
    (all_annot_eco["label_category"] == "Orthoptera") &
    (all_annot_eco["label"].isin(UK_SPECIES))
].copy()

# Search recursively within entire ecosoundset directory to build file index.
# Removes need to assume directory/subdirectory structure.
eco_file_index = {file_path.name: file_path for file_path in ECO_ROOT.rglob("*.wav")}
orth_eco["filepath"] = orth_eco["audio_segment_file_name"].map(eco_file_index)


orth_eco.rename(columns={
    "label": "species",
    "audio_segment_initial_time": "t_min",
    "audio_segment_final_time":   "t_max",
}, inplace=True)

eco_train = orth_eco[orth_eco["subset"] == "train"]
eco_val   = orth_eco[orth_eco["subset"] == "val"]
eco_test  = orth_eco[orth_eco["subset"] == "test"]
print(f"ECOSoundSet — train: {len(eco_train)}  val: {len(eco_val)}  test: {len(eco_test)}")
eco_train["species"].value_counts()
print(eco_test[eco_test["filepath"].isna()])

ECOSoundSet — train: 15621  val: 6922  test: 6459
        recording_id  t_min  t_max  annotation_initial_time  \
96806          43655      0      4                 0.000000   
96808          43655     12     16                12.000000   
96809          43655     16     20                17.867672   
96810          43655     16     20                18.760737   
96811          43655     16     20                18.567228   
...              ...    ...    ...                      ...   
111156         44803    632    636               632.387845   
111327         48510      0      4                 0.000000   
111328         48510     20     24                21.178618   
111330         48510     24     28                24.000000   
111331         48510    160    164               160.000000   

        annotation_final_time  annotation_min_freq  annotation_max_freq  \
96806                4.000000          8574.046000         48000.000000   
96808               16.000000          8427

In [6]:
# ── Supplement with InsectSet459 (train and validation splits) ──────────────────────────
# InsectSet459 has no test split — we use it only to top up training data.
all_annot_insect = pd.read_csv(INSECT_ROOT / "InsectSet459_Train_Val_Annotation.csv")

# Map matching species_names to canonical counterpart in INSECT_TO_ECO.
# .map() returns NaN for any key (species_name) not in INSECT_TO_ECO.
all_annot_insect["species"] = all_annot_insect["species_name"].map(INSECT_TO_ECO)
insect_train = all_annot_insect[all_annot_insect["species"].notna()].copy()

# Search recursively within entire insectset459 directory to build file index.
# Removes need to assume directory/subdirectory structure.
insect_file_index = {f.name: f for f in INSECT_ROOT.rglob("*") if f.suffix in (".wav", ".mp3")}
insect_train["filepath"] = insect_train["file_name"].map(insect_file_index)

insect_train["t_min"] = 0.0
insect_train["t_max"] = 4.0  # InsectSet459 clips are typically ~4 s

print(f"InsectSet459 — train (train + validation subsets): {len(insect_train)}")
insect_train["species"].value_counts()


InsectSet459 — train (train + validation subsets): 2018


species
Gryllus campestris                         540
Tettigonia viridissima                     497
Roeseliana roeselii                        233
Pholidoptera griseoaptera                  227
Pseudochorthippus parallelus parallelus    181
Chorthippus brunneus brunneus              164
Leptophyes punctatissima                   106
Omocestus viridulus                         70
Name: count, dtype: int64

In [26]:
# ── Merge and write OpenSoundscape one-hot CSVs ──────────────────────────────
# OSS expects index (file, start_time, end_time) + one-hot species columns.

def to_oss_format(df):
    # Group by clip to handle multiple species annotations per clip (relevant for ECOSoundSet).
    # Ensures one-hot encoded cols indicate all species that appear in the clip.
    grouped = df.groupby(["filepath", "t_min", "t_max"])["species"].apply(set).reset_index()
    
    rows = []
    for _, row in grouped.iterrows():
        entry = {
            "file":       row["filepath"],
            "start_time": 0, # every EcoSoundSet clip is 4 seconds, t_min and t_max are relative to whole clip, not split.
            "end_time":   4,
        }
        for sp in UK_SPECIES:
            entry[sp] = 1 if sp in row["species"] else 0
        rows.append(entry)
    return pd.DataFrame(rows).set_index(["file", "start_time", "end_time"])

# Combine ECO + InsectSet459 for training; use ECO only for val/test
combined_train = pd.concat([eco_train, insect_train], ignore_index=True)

train_oss = to_oss_format(combined_train)
val_oss   = to_oss_format(eco_val)
test_oss  = to_oss_format(eco_test)

train_oss.to_csv(OUT_DIR / "train.csv")
val_oss.to_csv(OUT_DIR / "val.csv")
test_oss.to_csv(OUT_DIR / "test.csv")

print(f"Saved to {OUT_DIR}/")
print(f"Train: {len(train_oss)}  Val: {len(val_oss)}  Test: {len(test_oss)}")
print("\nPer-species train counts:")
print(train_oss.sum().sort_values(ascending=False).to_string())


Saved to /Users/dawsonmccall/Documents/coding_projects/BioAcousticStreamEngine/training/data/
Train: 10818  Val: 3602  Test: 2768

Per-species train counts:
Gryllus campestris                         3428
Tettigonia viridissima                     2888
Roeseliana roeselii                        1993
Pseudochorthippus parallelus parallelus    1073
Pholidoptera griseoaptera                   952
Leptophyes punctatissima                    719
Chorthippus brunneus brunneus               269
Omocestus viridulus                         198
